# Test divide-and-conquer subgraph isomorphism

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

# import qiskit
# print(qiskit.__qiskit_version__)
    
from qiskit.transpiler import CouplingMap
from qiskit import QuantumCircuit, QuantumRegister
from vfs import Vf

import networkx as nx
# from qiskit.tools.parallel import CPU_COUNT
# print('CPU Count: ', CPU_COUNT)

def is_embeddable(g, H, anchor, stop):
    '''check if a small graph g is embeddable in a large H, anchor is bool
        g, H (Graph)
        anchor (bool): whether or not mapping anchor of g to that of H
        stop (float): time limit for vf2
    '''
    vf2 = Vf()
    result = {} 
    if anchor: result[hub(g)] = hub(H)
    result = vf2.dfsMatch(g, H, result, stop)
    lng = len(nx.nodes(g))
    if len(result) == lng:
        return True, result   
    return False, result


def extract_circuit(filename, path, node_num):
    # compose the input Quantum Circuit
    q = QuantumRegister(node_num, 'q')
    cir_in = QuantumCircuit(q)
    cir_temp = cir_in.from_qasm_file(path+filename)
    cir_in.compose(cir_temp, inplace=True)
    if node_num < cir_temp.num_qubits:
        raise Exception("Cannot compose circuit!") 

    return cir_in

def graph_of_circuit(circuit):
    ''' Return the induced graph of the reduced circuit C
            - node set: qubits in C
            - edge set: all pair (p,q) if CNOT [p,q] or CNOT[q,p] in C
        Args:
            C (list): the input reduced circuit
        Returns:
            g (Graph)
    '''   
    g = nx.Graph()
    for gate in circuit:
        if gate.operation.name != 'cx': continue
        p, q = gate.qubits[0]._index, gate.qubits[1]._index
        g.add_edge(p,q)
    return g  

print('test')

test


In [8]:
import ag
import time
import os
"""AG and Benchmarks"""

#AG = ag.qgrid(2,3)
#AG_name = 'G2x3'
path = '../bench/6Qbench/'

AG = ag.q20() 

# import networkx as nx
import matplotlib.pyplot as plt


coupling_map = CouplingMap(couplinglist=AG.edges())

start = time.time()    
print(time.asctime())


count = 0
for filename in os.listdir(path):
    if not filename.endswith('.qasm'): continue
    if filename != '6QBT_large_gate_opt2_3_1.5_no.7.qasm': continue
    print(filename)

    '''Extract the circuit from qasm files'''
    circuit = extract_circuit(filename, path, coupling_map.size())
    print(circuit.depth(), circuit.count_ops(), circuit.size())
    g = graph_of_circuit(circuit)
    #print(g.edges())
    print(is_embeddable(g,AG,False,10))

end = time.time()
print('Used time (s):', round(end-start,2), time.asctime() )


Sun Sep 10 14:13:04 2023
6QBT_large_gate_opt2_3_1.5_no.7.qasm
14 OrderedDict([('h', 18), ('cx', 12)]) 30
(True, {0: 1, 3: 2, 4: 6, 1: 7, 2: 3, 5: 4})
Used time (s): 0.0 Sun Sep 10 14:13:04 2023
